In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# ==========================================
# STEP 1: LOAD THE DATA
# ==========================================
# In a Kaggle comp, 'train' has the answers, 'test' does not.
print("Loading data...")
train = pd.read_csv("./datasets/train.csv")
test = pd.read_csv("./datasets/test.csv")

# ==========================================
# STEP 2: SEPARATE FEATURES (X) AND TARGET (y)
# ==========================================
# We drop 'Date' because raw dates confuse ML models.
# We drop 'Target' from X because it's the answer we are trying to predict.
X = train.drop(columns=["Date", "Target"])
y = train["Target"]

# ==========================================
# STEP 3: THE TIME-SERIES SPLIT (CRITICAL)
# ==========================================
# shuffle=False guarantees we train on the PAST and validate on the FUTURE.
# If you shuffle, you will accidentally cheat, get 99% accuracy locally,
# and fail the actual competition.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)

# ==========================================
# STEP 4: TRAIN THE LAPTOP-OPTIMIZED MODEL
# ==========================================
# LightGBM bins continuous features into histograms, making it insanely
# fast on a CPU compared to Random Forest. n_jobs=-1 uses all CPU cores.
print("Training LightGBM model...")
model = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

# ==========================================
# STEP 5: EVALUATE LOCAL PERFORMANCE
# ==========================================
# This is your true north. Trust this number more than the Kaggle scoreboard.
val_predictions = model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)
print(f"Local Validation Accuracy: {val_accuracy:.4f}")

# ==========================================
# STEP 6: PREDICT THE UNSEEN COMPETITION DATA
# ==========================================
# The test set must have the exact same columns as X_train.
X_test = test.drop(columns=["Date"])
test_predictions = model.predict(X_test)

# ==========================================
# STEP 7: FORMAT AND SAVE FOR KAGGLE
# ==========================================
# Kaggle requires a very specific format. Usually an ID/Date column and your prediction.
submission = pd.DataFrame({"Date": test["Date"], "Target": test_predictions})

# index=False is vital. If you leave it out, pandas adds a column of row numbers
# and the Kaggle automated grader will immediately reject your file.
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv! Ready to drag and drop.")

AttributeError: partially initialized module 'lightgbm' has no attribute 'LGBMClassifier' (most likely due to a circular import)